# AI for IoT - Practical 1: Fire Alarm Detection

**Objective:** Build a binary classifier that predicts whether an IoT smoke detector's alarm is ON (`1`) or OFF (`0`) from environmental sensor readings.

**Models:**
1. Logistic Regression (core task, baseline)
2. XGBoost (bonus challenge)

**Dataset:** Smoke Detection Dataset (Kaggle, `deepcontractor/smoke-detection-dataset`)

**Pipeline:** Load -> Inspect -> Split -> Scale -> Train -> Evaluate -> Compare -> Conclude

## 1. Imports

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from xgboost import XGBClassifier

sns.set_style("whitegrid")
RANDOM_STATE = 42

## 2. Load and Inspect the Data

In [ ]:
# Locate the CSV automatically inside the Kaggle input folder
path = glob.glob('/kaggle/input/**/smoke_detection_iot.csv', recursive=True)[0]
df = pd.read_csv(path)
print("Shape:", df.shape)
df.head()

In [ ]:
# Data types and non-null counts
df.info()

# Missing values per column
print("\nMissing values:\n", df.isnull().sum())

# Duplicated rows
print("\nDuplicated rows:", df.duplicated().sum())

# Basic statistics
df.describe().T

**Observation:** If no missing values are reported, no imputation or removal is needed.
The columns `Unnamed: 0` (row index), `UTC` (timestamp) and `CNT` (sample counter) are identifiers, not sensor measurements, so they are dropped to avoid the model learning from them.

## 3. Class Balance and Correlations

In [ ]:
# Class distribution of the target
counts = df['Fire Alarm'].value_counts()
print(counts)
print("\nProportions:\n", df['Fire Alarm'].value_counts(normalize=True).round(3))

plt.figure(figsize=(5, 4))
sns.countplot(x='Fire Alarm', hue='Fire Alarm', data=df, palette='Set2', legend=False)
plt.title('Class Distribution (0 = Normal, 1 = Fire Alarm)')
plt.show()

In [ ]:
# Correlation matrix of sensor features and the target
drop_cols = ['Unnamed: 0', 'UTC', 'CNT']
corr = df.drop(columns=drop_cols).corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.show()

## 4. Features/Target, Train-Test Split, Scaling

The split is done **before** scaling and the scaler is fitted on the training set only, to prevent **data leakage** from the test set.

In [ ]:
# Define features (X) and target (y), dropping identifier columns
X = df.drop(columns=['Unnamed: 0', 'UTC', 'CNT', 'Fire Alarm'])
y = df['Fire Alarm']
print("Features:", list(X.columns))

# 80% train / 20% test, stratified to keep the class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

# Feature scaling (mandatory for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train only
X_test_scaled = scaler.transform(X_test)         # reuse train statistics

print("Train:", X_train.shape, "| Test:", X_test.shape)

In [ ]:
# Helper functions reused for both models
def evaluate(name, y_true, y_pred):
    """Compute and print the four required metrics, return them as a dict."""
    res = {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
    }
    for k, v in res.items():
        if k != 'Model':
            print(f"{k:10s}: {v:.4f}")
    print("\n", classification_report(y_true, y_pred, target_names=['Normal', 'Fire Alarm']))
    return res

def plot_cm(y_true, y_pred, title, cmap='Blues'):
    """Plot the confusion matrix as an annotated heatmap."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=['Normal', 'Fire Alarm'],
                yticklabels=['Normal', 'Fire Alarm'])
    plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title(title)
    plt.show()
    tn, fp, fn, tp = cm.ravel()
    print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

## 5. Core Task: Logistic Regression

In [ ]:
# Train
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_scaled, y_train)

# Predict
y_pred_lr = log_reg.predict(X_test_scaled)

# Evaluate
res_lr = evaluate('Logistic Regression', y_test, y_pred_lr)
plot_cm(y_test, y_pred_lr, 'Confusion Matrix - Logistic Regression', 'Blues')

## 6. Bonus: XGBoost

In [ ]:
import re

# XGBoost does not accept [, ] or < in feature names -> clean them (copies only)
def clean_name(c):
    return re.sub(r'[\[\]<%]', '', c).replace(' ', '_')

X_train_xgb = X_train.copy()
X_test_xgb = X_test.copy()
X_train_xgb.columns = [clean_name(c) for c in X_train.columns]
X_test_xgb.columns = [clean_name(c) for c in X_test.columns]

# Train on the same train/test split (trees are insensitive to scaling,
# so the raw features are used)
xgb = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                    eval_metric='logloss', random_state=RANDOM_STATE)
xgb.fit(X_train_xgb, y_train)

# Predict and evaluate
y_pred_xgb = xgb.predict(X_test_xgb)
res_xgb = evaluate('XGBoost', y_test, y_pred_xgb)
plot_cm(y_test, y_pred_xgb, 'Confusion Matrix - XGBoost', 'Greens')

In [ ]:
# Feature importance (which sensors matter most)
imp = pd.Series(xgb.feature_importances_, index=X_train_xgb.columns).sort_values()
imp.plot(kind='barh', figsize=(8, 5), color='seagreen')
plt.title('XGBoost Feature Importance')
plt.xlabel('Importance')
plt.show()

## 7. Model Comparison

In [ ]:
comparison = pd.DataFrame([res_lr, res_xgb]).set_index('Model').round(4)
display(comparison)

ax = comparison.plot(kind='bar', figsize=(9, 5), ylim=(0.80, 1.005))
plt.title('Logistic Regression vs XGBoost - random split (y-axis starts at 0.80)')
plt.ylabel('Score'); plt.xticks(rotation=0)
plt.legend(loc='upper left', bbox_to_anchor=(1.01, 1))
for c in ax.containers:
    ax.bar_label(c, fmt='%.3f', fontsize=8, padding=2)
plt.tight_layout()
plt.show()

## 7b. Robustness Check: Chronological Split

The random 80/20 split gave near-perfect XGBoost scores. Because the samples are 1-second time-series readings, neighbouring samples are almost identical, so a random split may leak information from train to test. Here the data is sorted by timestamp and the **last 20% in time** is used as the test set, which better simulates deployment on future, unseen data.

In [ ]:
# Sort chronologically using the timestamp column
df_time = df.sort_values('UTC').reset_index(drop=True)
feature_cols = [c for c in df_time.columns if c not in ['Unnamed: 0', 'UTC', 'CNT', 'Fire Alarm']]
X_t = df_time[feature_cols]
y_t = df_time['Fire Alarm']

# Chronological 80/20 split (first 80% = train, last 20% = test, NO shuffling)
split_idx = int(len(df_time) * 0.8)
Xt_train, Xt_test = X_t.iloc[:split_idx], X_t.iloc[split_idx:]
yt_train, yt_test = y_t.iloc[:split_idx], y_t.iloc[split_idx:]

print("Train class distribution:", yt_train.value_counts(normalize=True).round(3).to_dict())
print("Test  class distribution:", yt_test.value_counts(normalize=True).round(3).to_dict())
print(f"Train: {Xt_train.shape} | Test: {Xt_test.shape}")

# Logistic Regression (scaler fitted on train only)
scaler_t = StandardScaler()
Xt_train_s = scaler_t.fit_transform(Xt_train)
Xt_test_s = scaler_t.transform(Xt_test)
lr_t = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_t.fit(Xt_train_s, yt_train)
pred_lr_t = lr_t.predict(Xt_test_s)

# XGBoost (cleaned feature names)
Xt_train_x = Xt_train.copy(); Xt_test_x = Xt_test.copy()
Xt_train_x.columns = [clean_name(c) for c in Xt_train.columns]
Xt_test_x.columns = [clean_name(c) for c in Xt_test.columns]
xgb_t = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                      eval_metric='logloss', random_state=RANDOM_STATE)
xgb_t.fit(Xt_train_x, yt_train)
pred_xgb_t = xgb_t.predict(Xt_test_x)

def metrics_row(name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {'Model': name,
            'Accuracy': accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred, zero_division=0),
            'Recall': recall_score(y_true, y_pred, zero_division=0),
            'F1-Score': f1_score(y_true, y_pred, zero_division=0),
            'FN': fn, 'FP': fp}

temporal = pd.DataFrame([metrics_row('Logistic Regression', yt_test, pred_lr_t),
                         metrics_row('XGBoost', yt_test, pred_xgb_t)]).set_index('Model')
print("\n=== Chronological split results ===")
display(temporal.round(4))

compare_split = pd.DataFrame({
    'Random split': [res_lr['Recall'], res_xgb['Recall']],
    'Chronological split': [temporal.loc['Logistic Regression', 'Recall'],
                            temporal.loc['XGBoost', 'Recall']],
}, index=['Logistic Regression', 'XGBoost']).round(4)
print("\n=== Recall: random vs chronological split ===")
display(compare_split)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, pred, cmap) in zip(axes, [('Logistic Regression', pred_lr_t, 'Blues'),
                                         ('XGBoost', pred_xgb_t, 'Greens')]):
    cm = confusion_matrix(yt_test, pred, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['Normal', 'Fire Alarm'], yticklabels=['Normal', 'Fire Alarm'])
    ax.set_title(f'{name} (chronological split)')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()

## 8. Conclusion

In [ ]:
# Summary of the key numbers
cm_lr = confusion_matrix(y_test, y_pred_lr).ravel()
cm_xgb = confusion_matrix(y_test, y_pred_xgb).ravel()
summary = pd.DataFrame({
    'Split': ['Random', 'Random', 'Chronological', 'Chronological'],
    'Model': ['Logistic Regression', 'XGBoost', 'Logistic Regression', 'XGBoost'],
    'Recall': [res_lr['Recall'], res_xgb['Recall'],
               temporal.loc['Logistic Regression', 'Recall'], temporal.loc['XGBoost', 'Recall']],
    'FN': [cm_lr[2], cm_xgb[2], temporal.loc['Logistic Regression', 'FN'], temporal.loc['XGBoost', 'FN']],
    'FP': [cm_lr[1], cm_xgb[1], temporal.loc['Logistic Regression', 'FP'], temporal.loc['XGBoost', 'FP']],
}).round(4)
display(summary)

### Logistic Regression - Recall interpretation
With the standard random split, Logistic Regression reaches a Recall of **0.9482**: it detects about 94.8% of real fire events (8,487 of 8,951) and misses 5.2% (**464 False Negatives**), while raising **850 False Positives**. Since Recall = TP / (TP + FN), it directly measures how reliable the alarm is when a real fire occurs. Roughly 1 real event in 19 goes undetected, which is acceptable for a baseline but not for a safety-critical device.

Note on Accuracy: the fire class is 71.5% of the data, so always predicting "fire" would already give 71.5% accuracy. Recall and F1 are more informative than Accuracy alone.

### Why False Negatives matter most
A False Negative is a real fire that goes undetected: no alert, delayed evacuation, risk to lives and property. A False Positive is only a false alarm, which is annoying and costly but not dangerous. In fire detection, Recall must therefore be prioritized over Precision.

### Which model has the higher Recall?
| Split | Model | Accuracy | Precision | Recall | F1 | FN | FP |
|---|---|---|---|---|---|---|---|
| Random 80/20 | Logistic Regression | 0.8951 | 0.9090 | 0.9482 | 0.9281 | 464 | 850 |
| Random 80/20 | **XGBoost** | 0.9999 | 1.0000 | **0.9999** | 0.9999 | **1** | 0 |
| Chronological 80/20 | **Logistic Regression** | 0.8892 | 0.9130 | **0.8781** | 0.8952 | **823** | 565 |
| Chronological 80/20 | XGBoost | 0.9105 | 1.0000 | 0.8340 | 0.9095 | 1121 | 0 |

- **Random split:** XGBoost has the higher Recall (0.9999 vs 0.9482), missing only 1 fire out of 8,951 with no false alarm. This shows the sensor-to-alarm relationship is strongly non-linear.
- **Chronological split:** the ranking on Recall **reverses**: Logistic Regression (0.8781) now detects more fires than XGBoost (0.8340), although XGBoost keeps a higher Accuracy (0.9105 vs 0.8892), a perfect Precision and zero false alarms.

### Effect of the evaluation protocol (temporal leakage)
The near-perfect random-split score does not hold on future data. XGBoost's Recall falls from **0.9999 to 0.8340** (1 to 1,121 missed fires) and Logistic Regression's from **0.9482 to 0.8781**. Because samples are recorded every second, a random split places almost identical neighbours of each test sample in the training set, so the model partly memorizes recording sessions. In addition, the class distribution shifts over time (fire class: 75.9% in train vs 53.9% in test), showing that the later period contains conditions not seen during training. XGBoost, being highly flexible, fits the training sessions very tightly and generalizes worse in Recall; it becomes conservative: it never raises a false alarm (Precision = 1.0) but classifies 1 in 6 real fires as normal, which is the worst failure mode for a fire alarm. The simpler linear model degrades more gracefully.

### Trade-offs
| | Logistic Regression | XGBoost |
|---|---|---|
| Simplicity | Very simple, interpretable coefficients | More complex, many hyperparameters |
| Speed / size | Very fast, tiny model, suits microcontrollers (edge) | Heavier, slower inference |
| Non-linear patterns | Limited (linear boundary) | Captures complex feature interactions |
| Preprocessing | Requires scaling | Insensitive to scaling |
| Random-split performance | Good baseline | Near-perfect (optimistic) |
| Robustness over time (Recall) | 0.878 | 0.834 |

### Implications for IoT fire alarm reliability
Under a realistic (chronological) evaluation, **both models miss more than 12% of real fires**, which is not acceptable for a safety-critical alarm. The very high random-split scores would have given a false sense of reliability. For deployment we recommend:
1. Evaluate with time-based or session-based splits, never with random splits on sensor time series.
2. Prioritize Recall: lower the decision threshold (e.g. 0.3 instead of 0.5) or use class weighting (`class_weight` / `scale_pos_weight`), accepting more false alarms.
3. Collect more diverse recording sessions (different rooms, seasons, fire sources) and retrain regularly to handle distribution shift.
4. On constrained devices Logistic Regression is a viable and, here, more Recall-robust option; XGBoost is preferable only once its generalization to unseen conditions is validated.

### Limitations
Results come from a single dataset recorded in a limited number of sessions and a single chronological split; hyperparameters were not tuned. A time-series cross-validation and threshold tuning would give a more reliable estimate of real-world performance.